# Screening Credit Agentic AI — Preprocessing, EDA & Scoring Pipeline (v2)

**Update v2:** dataset generator sekarang menyertakan 3 kolom baru di
`retail_customer_profile`: `jenis_kredit_diajukan`, `tenor_diajukan_bulan`,
`tujuan_penggunaan_kredit` (input pengajuan debitur), dan `estimated_dsr`
sudah dihitung dari tenor & bunga yang BENERAN diajukan (bukan asumsi
hardcode 36 bulan/12% seperti versi sebelumnya).

Notebook ini mencakup:
1. Load & inspeksi 8 tabel mentah
2. Join & preprocessing → `master_dataset.csv`
3. Feature engineering (fitur turunan DSR & pengajuan kredit)
4. EDA — univariate, bivariate, korelasi (Plotly, semua chart berlabel)
5. Agentic Scoring Pipeline (7 agent) + kategorisasi kelayakan 4-level
6. Export `master_scored.csv`


## 0. Setup

In [1]:
!pip install -q plotly


In [2]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

pd.set_option("display.max_columns", 100)


## 1. Load Data Mentah

Ganti `RAW_DIR` sesuai lokasi 8 CSV hasil `dataset_generator_v2.py` kamu
(`/content/dataset` kalau di Colab, atau folder lokal kamu).

In [3]:
RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

for file in sorted(os.listdir(RAW_DIR)):
    path = os.path.join(RAW_DIR, file)
    if file.endswith(".csv"):
        df_preview = pd.read_csv(path, nrows=3)
        print(f"{file:32s} | shape awal (preview 3 baris) | kolom: {list(df_preview.columns)}")


agunan_atr_bpn.csv               | shape awal (preview 3 baris) | kolom: ['atr_bpn_id', 'NIK', 'asset_type', 'certificate_type', 'certificate_number', 'provinsi', 'kota', 'kecamatan', 'kelurahan', 'land_area_m2', 'building_area_m2', 'nilai_tanah_per_m2', 'nilai_bangunan_per_m2', 'nilai_tanah_total', 'nilai_bangunan_total', 'total_collateral_value', 'ownership_match']
bank_account.csv                 | shape awal (preview 3 baris) | kolom: ['account_id', 'NIK', 'account_number', 'bank_name', 'account_type', 'account_status', 'opened_date', 'average_balance_6m', 'average_monthly_credit', 'average_monthly_debit', 'transaction_frequency_monthly', 'overdraft_count_6m', 'current_balance']
dhn.csv                          | shape awal (preview 3 baris) | kolom: ['dhn_id', 'NIK', 'status_dhn', 'alasan', 'tanggal_input']
dukcapil.csv                     | shape awal (preview 3 baris) | kolom: ['dukcapil_id', 'NIK', 'nama', 'tempat_lahir', 'tanggal_lahir', 'jenis_kelamin', 'golongan_darah', 'ala

## 2. Join & Preprocessing → `master_dataset.csv`

Sama seperti pipeline sebelumnya (agregasi tabel 1:banyak terhadap NIK,
`rm_master` di-join 1:1 lewat `rm_id`), TIDAK ada yang diubah di logic
join — 3 kolom pengajuan kredit baru (`jenis_kredit_diajukan`,
`tenor_diajukan_bulan`, `tujuan_penggunaan_kredit`) otomatis ikut terbawa
karena sudah ada di `retail_customer_profile.csv` sejak sumbernya.

In [4]:
NIK_STR = {"NIK": str}

profile  = pd.read_csv(f"{RAW_DIR}/retail_customer_profile.csv", dtype=NIK_STR)
dukcapil = pd.read_csv(f"{RAW_DIR}/dukcapil.csv", dtype=NIK_STR)
slik     = pd.read_csv(f"{RAW_DIR}/slik_credit_history.csv", dtype=NIK_STR)
dhn      = pd.read_csv(f"{RAW_DIR}/dhn.csv", dtype=NIK_STR)
agunan   = pd.read_csv(f"{RAW_DIR}/agunan_atr_bpn.csv", dtype=NIK_STR)
fin      = pd.read_csv(f"{RAW_DIR}/laporan_keuangan.csv", dtype=NIK_STR)
bank     = pd.read_csv(f"{RAW_DIR}/bank_account.csv", dtype={**NIK_STR, "account_number": str})
rm       = pd.read_csv(f"{RAW_DIR}/rm_master.csv")

# --- Agregasi tabel 1:banyak terhadap NIK ---
slik_agg = (slik.groupby("NIK")
    .agg(slik_n_loans=("slik_record_id","count"),
         slik_worst_collectability=("collectability","max"),
         slik_n_banks=("bank_name","nunique"),
         slik_total_outstanding=("outstanding_balance","sum"),
         slik_total_installment_other=("installment_amount","sum"),
         slik_avg_tenor_month=("tenor_month","mean"))
    .reset_index())
slik_agg["slik_has_macet"] = (slik_agg["slik_worst_collectability"] == 5).astype(int)
slik_agg["slik_has_credit_history"] = 1

bank_agg = (bank.groupby("NIK")
    .agg(bank_n_accounts=("account_id","count"),
         bank_best_avg_balance_6m=("average_balance_6m","max"),
         bank_total_avg_credit=("average_monthly_credit","sum"),
         bank_total_avg_debit=("average_monthly_debit","sum"),
         bank_total_overdraft_6m=("overdraft_count_6m","sum"),
         bank_current_balance_total=("current_balance","sum"),
         bank_best_current_balance=("current_balance","max"))
    .reset_index())
bank_agg["bank_any_dormant"] = bank.groupby("NIK")["account_status"] \
    .apply(lambda s: int((s == "Dormant").any())).values

fin_pivot = fin.pivot_table(index="NIK", columns="year",
    values=["revenue","net_profit","total_asset","total_liability","operating_cashflow"])
fin_pivot.columns = [f"{col}_{yr}" for col, yr in fin_pivot.columns]
fin_pivot = fin_pivot.reset_index()
fin_pivot["revenue_growth_pct"] = ((fin_pivot["revenue_2025"] - fin_pivot["revenue_2024"])
                                     / fin_pivot["revenue_2024"]).round(4)
fin_pivot["profit_margin_2025"] = (fin_pivot["net_profit_2025"] / fin_pivot["revenue_2025"]).round(4)
fin_pivot["liability_to_asset_2025"] = (fin_pivot["total_liability_2025"]
                                          / fin_pivot["total_asset_2025"]).round(4)

agunan_extra = agunan[["NIK","kelurahan","land_area_m2","building_area_m2",
                        "nilai_tanah_per_m2","nilai_bangunan_per_m2"]].rename(
    columns={"kelurahan":"agunan_kelurahan"})
dhn_slim = dhn[["NIK","status_dhn","alasan"]].rename(columns={"alasan":"dhn_alasan"})

# rm_master: relasi 1:1 thd rm_id (BUKAN NIK) - LEFT JOIN biasa, tidak diagregasi
rm_slim = rm.rename(columns={"branch_name": "rm_branch_name", "region": "rm_region"})

# --- Join semuanya ke master table ---
master = (profile
    .merge(dhn_slim, on="NIK", how="left")
    .merge(slik_agg, on="NIK", how="left")
    .merge(bank_agg, on="NIK", how="left")
    .merge(fin_pivot, on="NIK", how="left")
    .merge(agunan_extra, on="NIK", how="left")
    .merge(rm_slim, on="rm_id", how="left"))

# --- Preprocessing dasar ---
slik_num_cols = ["slik_n_loans","slik_worst_collectability","slik_n_banks",
                  "slik_total_outstanding","slik_total_installment_other",
                  "slik_avg_tenor_month","slik_has_macet"]
master[slik_num_cols] = master[slik_num_cols].fillna(0)
master["slik_has_credit_history"] = master["slik_has_credit_history"].fillna(0).astype(int)
master["dhn_alasan"] = master["dhn_alasan"].fillna("Tidak Berlaku")
master["application_date"] = pd.to_datetime(master["application_date"])

master["dsr_capped"] = master["estimated_dsr"].clip(upper=3.0)
master["is_female_owner"] = (master["owner_gender"] == "P").astype(int)
master["has_dhn_flag"] = (master["status_dhn"] == "Ya").astype(int)

missing = master.isna().sum()
missing = missing[missing > 0]
print("Kolom yang masih missing setelah preprocessing:", list(missing.index) if len(missing) else "(tidak ada)")

LEAKAGE_COLS = ["eligibility_score"]
SENSITIVE_COLS = ["owner_gender", "is_female_owner", "owner_marital_status"]
OPERATIONAL_ONLY_COLS = ["rm_id", "rm_name", "rm_branch_name", "rm_region", "jabatan", "level", "join_date"]
print(f"Kolom leakage (WAJIB drop sblm modeling): {LEAKAGE_COLS}")
print(f"Kolom sensitif (drop dari fitur model, fair-lending): {SENSITIVE_COLS}")
print(f"Kolom operasional RM (drop dari fitur model, governance): {OPERATIONAL_ONLY_COLS}")

print(f"\nMaster table (sebelum feature engineering): {master.shape[0]} baris x {master.shape[1]} kolom")
master.head(3)


Kolom yang masih missing setelah preprocessing: (tidak ada)
Kolom leakage (WAJIB drop sblm modeling): ['eligibility_score']
Kolom sensitif (drop dari fitur model, fair-lending): ['owner_gender', 'is_female_owner', 'owner_marital_status']
Kolom operasional RM (drop dari fitur model, governance): ['rm_id', 'rm_name', 'rm_branch_name', 'rm_region', 'jabatan', 'level', 'join_date']

Master table (sebelum feature engineering): 3000 baris x 86 kolom


,application_id,NIK,cif_number,application_date,customer_type,company_name,legal_entity,owner_name,owner_gender,owner_age,owner_marital_status,owner_education,province,city,district,region,branch_name,industry,sub_industry,business_age_year,employee_count,monthly_turnover_est,transaction_frequency_monthly,loan_requested,jenis_kredit_diajukan,tenor_diajukan_bulan,tujuan_penggunaan_kredit,collateral_type,collateral_location,collateral_province,collateral_city,collateral_size_m2,collateral_market_value,collateral_liquidation_value,collateral_ratio,certificate_type,ownership_match,estimated_dsr,eligibility_score,label,rm_id,status_dhn,dhn_alasan,slik_n_loans,slik_worst_collectability,slik_n_banks,slik_total_outstanding,slik_total_installment_other,slik_avg_tenor_month,slik_has_macet,slik_has_credit_history,bank_n_accounts,bank_best_avg_balance_6m,bank_total_avg_credit,bank_total_avg_debit,bank_total_overdraft_6m,bank_current_balance_total,bank_best_current_balance,bank_any_dormant,net_profit_2024,net_profit_2025,operating_cashflow_2024,operating_cashflow_2025,revenue_2024,revenue_2025,total_asset_2024,total_asset_2025,total_liability_2024,total_liability_2025,revenue_growth_pct,profit_margin_2025,liability_to_asset_2025,agunan_kelurahan,land_area_m2,building_area_m2,nilai_tanah_per_m2,nilai_bangunan_per_m2,rm_name,rm_branch_name,rm_region,jabatan,level,join_date,dsr_capped,is_female_owner,has_dhn_flag
0,APP202600001,3276010601750001,CIF1000001,2025-07-08,UMKM,UD Santoso Abadi,UD,Budi Panjaitan,L,51,Menikah,S2,Jawa Barat,Depok,Sukmajaya,Region 2,KCP Bogor Baranangsiang,Manufaktur,Konveksi,14,9,2672137,66,300000000,KI,54,Pembelian mesin produksi tambahan usaha Konveksi,Rumah,"Poris Plawad, Tangerang",Banten,Tangerang,423.4,2328496000,1862796800,7.76,HGB,Ya,3.0,0.804,Diterima,RM0020,Tidak,Tidak Berlaku,1.0,1.0,1.0,87078357.0,2882804.0,36.0,0.0,1,2,19800614,4408782,3829344,0,22715525,13581790,0,3405584.0,3318494.0,3039455.0,4499938.0,29666491.0,32065651.0,48148218.0,37084511.0,27015813.0,18787312.0,0.0809,0.1035,0.5066,Poris Plawad,187.3,236.1,8020000,3500000,Indah Kusuma,KCP Bogor Baranangsiang,Region 4,Relationship Banking Officer,Junior RB,2017-04-09,3.0,0,0
1,APP202600002,3172010301920002,CIF1000002,2025-09-01,UMKM,UD Wijaya Mandiri,UD,Andi Hidayat,L,34,Cerai Hidup,S2,DKI Jakarta,Jakarta Utara,Kramat Jati,Region 1,KCP Bekasi Barat,Jasa,Bengkel,1,3,1807917,39,75000000,KUR,24,Tambahan modal kerja usaha Bengkel,Rumah,"Pluit, Jakarta Utara",DKI Jakarta,Jakarta Utara,174.8,3724038000,2979230400,49.65,HGB,Ya,3.0,0.683,Diterima,RM0010,Tidak,Tidak Berlaku,2.0,2.0,2.0,219070769.0,16155953.0,24.0,0.0,1,1,4014256,1124770,1040039,0,1718267,1718267,0,2907631.0,2602253.0,3572130.0,2863796.0,22347316.0,21695009.0,27256315.0,35323008.0,17484128.0,9380894.0,-0.0292,0.1199,0.2656,Pluit,113.0,61.8,29970000,5460000,Indah Rahman,KCP Bekasi Barat,Region 3,Relationship Banking Officer,Senior RB,2022-07-08,3.0,0,0
2,APP202600003,3671010604800003,CIF1000003,2025-10-27,UMKM,UD Kusuma Sejahtera,CV,Doni Pratama,L,46,Cerai Hidup,S2,Banten,Tangerang,Bekasi Timur,Region 3,KCP Cibubur,Transportasi,Ekspedisi Kecil,17,33,1698736,59,150000000,KUR,24,Tambahan modal kerja usaha Ekspedisi Kecil,Tanah,"Sukabumi Selatan, Jakarta Barat",DKI Jakarta,Jakarta Barat,67.0,1681700000,1345360000,11.21,HGB,Ya,3.0,0.435,Ditolak,RM0039,Tidak,Tidak Berlaku,2.0,4.0,2.0,135687676.0,15096128.0,24.0,0.0,1,1,14489012,1394768,1222864,1,3156322,3156322,1,5615079.0,3670504.0,7404291.0,4126254.0,23914503.0,20384842.0,34076107.0,37104928.0,17322819.0,11862908.0,-0.1476,0.1801,0.3197,Sukabumi Selatan,67.0,0.0,25100000,5500000,Slamet Hutapea,KCP Cibubur,Region 2,Relationship Banking Officer,Senior RB,2021-10-25,3.0,0,0


## 3. Feature Engineering

Fitur turunan tambahan seputar DSR & struktur pengajuan kredit — belum
ada di tabel mentah, dihitung dari kolom yang sudah ada (termasuk 3
kolom baru `jenis_kredit_diajukan`, `tenor_diajukan_bulan`,
`tujuan_penggunaan_kredit`).

In [5]:
# --- Bunga per jenis kredit (sama seperti dipakai generator, dipakai lagi
# di sini untuk menghitung ulang cicilan sbg fitur eksplisit) ---
INTEREST_RATE = {"KUR": 0.06, "KMK": 0.11, "KI": 0.10}

master["bunga_persen_diajukan"] = master["jenis_kredit_diajukan"].map(INTEREST_RATE) * 100

# Cicilan bulanan estimasi dari pengajuan BARU ini saja (bukan total cicilan
# existing + baru seperti estimated_dsr) - berguna untuk lihat beban murni
# dari pengajuan yang sedang diproses
master["cicilan_bulanan_pengajuan"] = (
    master["loan_requested"] / master["tenor_diajukan_bulan"]
    * (1 + master["jenis_kredit_diajukan"].map(INTEREST_RATE) * master["tenor_diajukan_bulan"] / 12)
).round(0)

# Rasio pengajuan terhadap omset tahunan (loan-to-income) - beda dari DSR
# (yang membandingkan ke cicilan), ini murni ukuran skala pengajuan vs bisnis
master["loan_to_annual_revenue"] = (
    master["loan_requested"] / (master["monthly_turnover_est"] * 12)
).round(3)

# Kategori tenor pengajuan (pendek/menengah/panjang) - memudahkan EDA/segmentasi
def kategori_tenor(bulan):
    if bulan <= 12:
        return "Pendek (<=12 bln)"
    elif bulan <= 36:
        return "Menengah (13-36 bln)"
    else:
        return "Panjang (>36 bln)"

master["kategori_tenor_diajukan"] = master["tenor_diajukan_bulan"].apply(kategori_tenor)

# Kategori nominal pengajuan (buat EDA/segmentasi, mengikuti plafon KUR sbg acuan)
def kategori_nominal(nominal):
    if nominal <= 50_000_000:
        return "<=50jt (KUR Mikro)"
    elif nominal <= 500_000_000:
        return "50-500jt (KUR Kecil/KMK/KI)"
    else:
        return ">500jt (KMK/KI saja)"

master["kategori_nominal_diajukan"] = master["loan_requested"].apply(kategori_nominal)

# Flag konsistensi sederhana: KUR tapi nominal di luar plafon KUR Kecil (harusnya
# TIDAK terjadi kalau generator benar, tapi berguna sbg data-quality check)
master["flag_kur_di_luar_plafon"] = (
    (master["jenis_kredit_diajukan"] == "KUR") & (master["loan_requested"] > 500_000_000)
).astype(int)
print("Cek data quality - baris KUR di luar plafon (harus 0):", master["flag_kur_di_luar_plafon"].sum())

master.to_csv(f"{PROCESSED_DIR}/master_dataset.csv", index=False)
print(f"\nMaster table (setelah feature engineering): {master.shape[0]} baris x {master.shape[1]} kolom")
print("Tersimpan:", f"{PROCESSED_DIR}/master_dataset.csv")
master[["loan_requested","jenis_kredit_diajukan","tenor_diajukan_bulan","bunga_persen_diajukan",
        "cicilan_bulanan_pengajuan","loan_to_annual_revenue","kategori_tenor_diajukan",
        "kategori_nominal_diajukan","estimated_dsr"]].head(5)


Cek data quality - baris KUR di luar plafon (harus 0): 0

Master table (setelah feature engineering): 3000 baris x 92 kolom
Tersimpan: ../data/processed/master_dataset.csv


,loan_requested,jenis_kredit_diajukan,tenor_diajukan_bulan,bunga_persen_diajukan,cicilan_bulanan_pengajuan,loan_to_annual_revenue,kategori_tenor_diajukan,kategori_nominal_diajukan,estimated_dsr
0,300000000,KI,54,10.0,8055556.0,9.356,Panjang (>36 bln),50-500jt (KUR Kecil/KMK/KI),3.0
1,75000000,KUR,24,6.0,3500000.0,3.457,Menengah (13-36 bln),50-500jt (KUR Kecil/KMK/KI),3.0
2,150000000,KUR,24,6.0,7000000.0,7.358,Menengah (13-36 bln),50-500jt (KUR Kecil/KMK/KI),3.0
3,200000000,KUR,24,6.0,9333333.0,14.683,Menengah (13-36 bln),50-500jt (KUR Kecil/KMK/KI),3.0
4,750000000,KI,54,10.0,20138889.0,72.441,Panjang (>36 bln),>500jt (KMK/KI saja),3.0


## 4. Exploratory Data Analysis (EDA)

### 4.1 Univariate — Distribusi Tiap Fitur Penting

**4.1.1 Distribusi Label (Target)**

In [6]:
label_counts = master["label"].value_counts().reset_index()
label_counts.columns = ["label", "jumlah"]

fig = px.pie(label_counts, names="label", values="jumlah", hole=0.45,
             color="label", color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
             title="Distribusi Label (Target)")
fig.update_traces(textinfo="percent+label")
fig.show()

pct_diterima = (master["label"]=="Diterima").mean()*100
print(f"Insight: {pct_diterima:.1f}% pengajuan berlabel Diterima, {100-pct_diterima:.1f}% Ditolak.")


Insight: 84.8% pengajuan berlabel Diterima, 15.2% Ditolak.


**4.1.2 Kolektibilitas SLIK Terburuk**

In [7]:
collect_label_map = {0: "Belum Ada Riwayat", 1: "Lancar", 2: "DPK",
                      3: "Kurang Lancar", 4: "Diragukan", 5: "Macet"}
order = ["Belum Ada Riwayat","Lancar","DPK","Kurang Lancar","Diragukan","Macet"]
vc = master["slik_worst_collectability"].map(collect_label_map).value_counts().reindex(order).reset_index()
vc.columns = ["kolektibilitas", "jumlah"]

fig = px.bar(vc, x="kolektibilitas", y="jumlah", text_auto=True,
             title="Distribusi Kolektibilitas SLIK Terburuk",
             labels={"kolektibilitas": "Kolektibilitas", "jumlah": "Jumlah Nasabah"})
fig.show()

vc["persen"] = (vc["jumlah"] / vc["jumlah"].sum() * 100).round(1)
print("Insight:")
print(vc.to_string(index=False))


Insight:
   kolektibilitas  jumlah  persen
Belum Ada Riwayat     427    14.2
           Lancar    1487    49.6
              DPK     490    16.3
    Kurang Lancar     301    10.0
        Diragukan     166     5.5
            Macet     129     4.3


**4.1.3 Status Daftar Hitam Nasional (DHN)**

In [8]:
vc = master["status_dhn"].value_counts().reset_index()
vc.columns = ["status_dhn", "jumlah"]

fig = px.bar(vc, x="status_dhn", y="jumlah", color="status_dhn", text_auto=True,
             color_discrete_map={"Ya": "#e74c3c", "Tidak": "#2ecc71"},
             title="Distribusi Status Daftar Hitam Nasional (DHN)",
             labels={"status_dhn": "Terdaftar DHN?", "jumlah": "Jumlah Nasabah"})
fig.show()

pct_dhn = (master["status_dhn"]=="Ya").mean()*100
n_dhn = (master["status_dhn"]=="Ya").sum()
print(f"Insight: {n_dhn} dari {len(master)} nasabah ({pct_dhn:.1f}%) terdaftar di DHN — "
      f"kelompok ini otomatis kena hard-rule 'Tidak Layak' di Risk Agent, terlepas dari skor lain.")


Insight: 147 dari 3000 nasabah (4.9%) terdaftar di DHN — kelompok ini otomatis kena hard-rule 'Tidak Layak' di Risk Agent, terlepas dari skor lain.


**4.1.4 Estimasi DSR (Debt Service Ratio) — sudah pakai tenor & bunga hasil pengajuan**

In [9]:
fig = px.histogram(master, x="estimated_dsr", nbins=40,
                    title="Distribusi Estimasi DSR (dihitung dari tenor & bunga hasil pengajuan)",
                    labels={"estimated_dsr": "Estimasi DSR"})
fig.add_vline(x=3.0, line_dash="dash", line_color="red", annotation_text="batas cap (3.0)")
fig.show()

print(f"Insight: median DSR {master['estimated_dsr'].median():.2f}, "
      f"{(master['estimated_dsr']>=1).mean()*100:.1f}% nasabah punya DSR >= 1 (cicilan >= omset bulanan).")


Insight: median DSR 3.00, 98.4% nasabah punya DSR >= 1 (cicilan >= omset bulanan).


**4.1.5 Collateral Ratio**

In [10]:
fig = px.histogram(master, x="collateral_ratio", nbins=60, log_y=True,
                    title="Distribusi Collateral Ratio (sumbu-y log scale)",
                    labels={"collateral_ratio": "Collateral Ratio (nilai agunan / pinjaman)"})
fig.show()

pct_below_1 = (master["collateral_ratio"] < 1).mean() * 100
print(f"Insight: median collateral ratio {master['collateral_ratio'].median():.2f}x, "
      f"{pct_below_1:.1f}% nasabah punya agunan bernilai LEBIH KECIL dari nominal yang diajukan (ratio < 1).")


Insight: median collateral ratio 18.82x, 0.6% nasabah punya agunan bernilai LEBIH KECIL dari nominal yang diajukan (ratio < 1).


**4.1.6 Pertumbuhan Omset 2024→2025**

In [11]:
fig = px.histogram(master, x="revenue_growth_pct", nbins=40,
                    title="Distribusi Pertumbuhan Omset 2024→2025",
                    labels={"revenue_growth_pct": "Pertumbuhan Omset (%)"})
fig.add_vline(x=0, line_dash="dash", line_color="gray", annotation_text="0% (stagnan)")
fig.show()

pct_negatif = (master["revenue_growth_pct"] < 0).mean() * 100
print(f"Insight: rata-rata pertumbuhan omset {master['revenue_growth_pct'].mean()*100:.1f}%, "
      f"{pct_negatif:.1f}% nasabah mengalami penurunan omset (growth negatif).")


Insight: rata-rata pertumbuhan omset 11.6%, 27.2% nasabah mengalami penurunan omset (growth negatif).


**4.1.7 Sektor Industri**

In [12]:
vc = master["industry"].value_counts().reset_index()
vc.columns = ["industry", "jumlah"]
vc["persen"] = (vc["jumlah"] / vc["jumlah"].sum() * 100).round(1)

fig = px.bar(vc, x="industry", y="jumlah", text_auto=True,
             title="Distribusi Sektor Industri",
             labels={"industry": "Sektor Industri", "jumlah": "Jumlah Nasabah"})
fig.show()

print("Insight — jumlah & persentase nasabah per sektor industri:")
print(vc.to_string(index=False))


Insight — jumlah & persentase nasabah per sektor industri:
    industry  jumlah  persen
Transportasi     523    17.4
   Pertanian     514    17.1
 Perdagangan     506    16.9
  Manufaktur     501    16.7
        Jasa     484    16.1
     Kuliner     472    15.7


**4.1.8 BARU — Jenis Kredit yang Diajukan (KMK / KUR / KI)**

In [13]:
vc = master["jenis_kredit_diajukan"].value_counts().reset_index()
vc.columns = ["jenis_kredit_diajukan", "jumlah"]

fig = px.bar(vc, x="jenis_kredit_diajukan", y="jumlah", color="jenis_kredit_diajukan", text_auto=True,
             title="Distribusi Jenis Kredit yang Diajukan",
             labels={"jenis_kredit_diajukan": "Jenis Kredit Diajukan", "jumlah": "Jumlah Nasabah"})
fig.show()

print("Insight — jumlah & persentase pengajuan per jenis kredit:")
print(vc.assign(persen=(vc["jumlah"]/vc["jumlah"].sum()*100).round(1)).to_string(index=False))


Insight — jumlah & persentase pengajuan per jenis kredit:
jenis_kredit_diajukan  jumlah  persen
                  KMK    1166    38.9
                   KI     992    33.1
                  KUR     842    28.1


**4.1.9 BARU — Nominal Kredit yang Diajukan (`loan_requested`)**

In [14]:
fig = px.histogram(master, x="loan_requested", nbins=40, color="jenis_kredit_diajukan",
                    title="Distribusi Nominal Kredit yang Diajukan, per Jenis Kredit",
                    labels={"loan_requested": "Nominal Diajukan (IDR)", "jenis_kredit_diajukan": "Jenis Kredit"})
fig.add_vline(x=50_000_000, line_dash="dash", line_color="gray", annotation_text="plafon KUR Mikro")
fig.add_vline(x=500_000_000, line_dash="dash", line_color="gray", annotation_text="plafon KUR Kecil")
fig.show()

print(f"Insight: median nominal diajukan Rp{master['loan_requested'].median():,.0f}, "
      f"rentang Rp{master['loan_requested'].min():,.0f} - Rp{master['loan_requested'].max():,.0f}.")


Insight: median nominal diajukan Rp200,000,000, rentang Rp50,000,000 - Rp1,000,000,000.


**4.1.10 BARU — Tenor yang Diajukan, per Jenis Kredit**

In [15]:
fig = px.box(master, x="jenis_kredit_diajukan", y="tenor_diajukan_bulan", color="jenis_kredit_diajukan",
             title="Sebaran Tenor yang Diajukan per Jenis Kredit",
             labels={"jenis_kredit_diajukan": "Jenis Kredit", "tenor_diajukan_bulan": "Tenor Diajukan (bulan)"})
fig.show()

print("Insight: rentang tenor berbeda jelas antar jenis kredit (sesuai desain — KUR/KMK jangka pendek, KI jangka panjang):")
print(master.groupby("jenis_kredit_diajukan")["tenor_diajukan_bulan"].agg(["min","median","max"]))


Insight: rentang tenor berbeda jelas antar jenis kredit (sesuai desain — KUR/KMK jangka pendek, KI jangka panjang):
                       min  median  max
jenis_kredit_diajukan                  
KI                      36    48.0   60
KMK                     12    18.0   24
KUR                     12    24.0   36


**4.1.11 Total Current Balance**

In [16]:
fig = px.histogram(master, x="bank_current_balance_total", nbins=50,
                    title="Distribusi Total Current Balance (semua rekening per nasabah)",
                    labels={"bank_current_balance_total": "Total Current Balance (IDR)"})
fig.show()

pct_negatif_saldo = (master["bank_current_balance_total"] < 0).mean() * 100
print(f"Insight: median total current balance Rp{master['bank_current_balance_total'].median():,.0f}, "
      f"{pct_negatif_saldo:.1f}% nasabah punya saldo real-time negatif (indikasi overdraft aktif).")


Insight: median total current balance Rp18,474,114, 1.1% nasabah punya saldo real-time negatif (indikasi overdraft aktif).


**4.1.12 Level RM yang Menangani**

In [17]:
vc = master["level"].value_counts().reset_index()
vc.columns = ["level", "jumlah"]
vc["persen"] = (vc["jumlah"] / vc["jumlah"].sum() * 100).round(1)

fig = px.bar(vc, x="level", y="jumlah", color="level", text_auto=True,
             title="Distribusi Nasabah berdasarkan Level RM yang Menangani",
             labels={"level": "Level RM", "jumlah": "Jumlah Nasabah"})
fig.show()

print("Insight:")
print(vc.to_string(index=False))


Insight:
    level  jumlah  persen
Junior RB    1793    59.8
Senior RB    1207    40.2


**4.1.13 Usia Pemilik Usaha**

In [18]:
fig = px.histogram(master, x="owner_age", nbins=30,
                    title="Distribusi Usia Pemilik Usaha",
                    labels={"owner_age": "Usia Pemilik (tahun)"})
fig.show()

print(f"Insight: median usia pemilik {master['owner_age'].median():.0f} tahun, "
      f"rentang {master['owner_age'].min()}-{master['owner_age'].max()} tahun.")


Insight: median usia pemilik 42 tahun, rentang 23-61 tahun.


### 4.2 Bivariate — Hubungan Tiap Fitur dengan Label

**4.2.1 Kolektibilitas SLIK vs Label**

In [19]:
collect_label_map = {0: "Belum Ada Riwayat", 1: "Lancar", 2: "DPK",
                      3: "Kurang Lancar", 4: "Diragukan", 5: "Macet"}
order = ["Belum Ada Riwayat","Lancar","DPK","Kurang Lancar","Diragukan","Macet"]
master["collectability_readable"] = master["slik_worst_collectability"].map(collect_label_map)

ct = pd.crosstab(master["collectability_readable"], master["label"], normalize="index") * 100
ct = ct.reindex(order).reset_index().melt(id_vars="collectability_readable", var_name="label", value_name="persen")
ct["persen"] = ct["persen"].round(1)

fig = px.bar(ct, x="collectability_readable", y="persen", color="label", barmode="stack", text="persen",
             color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
             title="Kolektibilitas SLIK vs Label (%)",
             labels={"collectability_readable": "Kolektibilitas SLIK Terburuk", "persen": "Persentase (%)"})
fig.show()

print("Insight: pola monoton jelas — makin buruk kolektibilitas, makin tinggi persentase Ditolak:")
print(ct.pivot(index="collectability_readable", columns="label", values="persen").reindex(order).to_string())


Insight: pola monoton jelas — makin buruk kolektibilitas, makin tinggi persentase Ditolak:
label                    Diterima  Ditolak
collectability_readable                   
Belum Ada Riwayat            99.1      0.9
Lancar                       99.3      0.7
DPK                          94.5      5.5
Kurang Lancar                53.8     46.2
Diragukan                    12.0     88.0
Macet                         0.0    100.0


**4.2.2 Status DHN vs Label**

In [20]:
ct = master.groupby(["status_dhn", "label"]).size().reset_index(name="jumlah")

fig = px.bar(ct, x="status_dhn", y="jumlah", color="label", barmode="group", text_auto=True,
             color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
             title="Status DHN vs Label",
             labels={"status_dhn": "Terdaftar DHN?", "jumlah": "Jumlah Nasabah"})
fig.show()

print("Insight: hampir seluruh nasabah dengan status DHN 'Ya' berlabel Ditolak (sesuai hard-rule Risk Agent):")
print(ct.pivot(index="status_dhn", columns="label", values="jumlah").fillna(0).astype(int).to_string())


Insight: hampir seluruh nasabah dengan status DHN 'Ya' berlabel Ditolak (sesuai hard-rule Risk Agent):
label       Diterima  Ditolak
status_dhn                   
Tidak           2540      313
Ya                 4      143


**4.2.3 Estimasi DSR vs Label**

In [21]:
fig = px.box(master, x="label", y="estimated_dsr", color="label",
             color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
             title="Estimasi DSR vs Label",
             labels={"estimated_dsr": "Estimasi DSR", "label": "Label"})
fig.show()

med_dsr = master.groupby("label")["estimated_dsr"].median()
print(f"Insight: median DSR kelompok Diterima {med_dsr.get('Diterima', float('nan')):.2f} vs "
      f"Ditolak {med_dsr.get('Ditolak', float('nan')):.2f} — perbedaan relatif kecil, DSR bukan pembeda "
      "paling kuat dibanding Character/Collateral.")


Insight: median DSR kelompok Diterima 3.00 vs Ditolak 3.00 — perbedaan relatif kecil, DSR bukan pembeda paling kuat dibanding Character/Collateral.


**4.2.4 Collateral Ratio vs Label**

In [22]:
fig = px.box(master, x="label", y="collateral_ratio", color="label",
             color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
             log_y=True,
             title="Collateral Ratio vs Label (log scale)",
             labels={"collateral_ratio": "Collateral Ratio", "label": "Label"})
fig.show()

med_col = master.groupby("label")["collateral_ratio"].median()
print(f"Insight: median collateral ratio Diterima {med_col.get('Diterima', float('nan')):.2f}x vs "
      f"Ditolak {med_col.get('Ditolak', float('nan')):.2f}x — nasabah Ditolak cenderung agunannya lebih kecil relatif thd pinjaman.")


Insight: median collateral ratio Diterima 18.68x vs Ditolak 20.02x — nasabah Ditolak cenderung agunannya lebih kecil relatif thd pinjaman.


**4.2.5 Sektor Industri vs Label**

In [23]:
ct = master.groupby(["industry", "label"]).size().reset_index(name="jumlah")

fig = px.bar(ct, x="industry", y="jumlah", color="label", barmode="group", text_auto=True,
             color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
             title="Sektor Industri vs Label",
             labels={"industry": "Sektor Industri", "jumlah": "Jumlah Nasabah"})
fig.show()

rate_industry = master.groupby("industry")["label"].apply(lambda s: (s=="Diterima").mean()*100).round(1).sort_values()
print("Insight — approval rate relatif mirip di semua sektor (%):")
print(rate_industry.to_string())


Insight — approval rate relatif mirip di semua sektor (%):
industry
Pertanian       80.9
Kuliner         82.4
Transportasi    83.0
Manufaktur      86.6
Jasa            87.4
Perdagangan     88.5


**4.2.6 Pertumbuhan Omset vs Label**

In [24]:
fig = px.histogram(master, x="revenue_growth_pct", color="label", barmode="overlay",
                    nbins=40, opacity=0.65,
                    color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
                    title="Pertumbuhan Omset vs Label",
                    labels={"revenue_growth_pct": "Pertumbuhan Omset (%)"})
fig.show()

med_growth = master.groupby("label")["revenue_growth_pct"].median()
print(f"Insight: median growth Diterima {med_growth.get('Diterima', float('nan'))*100:.1f}% vs "
      f"Ditolak {med_growth.get('Ditolak', float('nan'))*100:.1f}% — kelompok Ditolak jelas condong ke pertumbuhan negatif.")


Insight: median growth Diterima 11.8% vs Ditolak 7.9% — kelompok Ditolak jelas condong ke pertumbuhan negatif.


**4.2.7 BARU — Jenis Kredit Diajukan vs Label**

In [25]:
ct = master.groupby(["jenis_kredit_diajukan", "label"]).size().reset_index(name="jumlah")

fig = px.bar(ct, x="jenis_kredit_diajukan", y="jumlah", color="label", barmode="group", text_auto=True,
             color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
             title="Jenis Kredit Diajukan vs Label",
             labels={"jenis_kredit_diajukan": "Jenis Kredit Diajukan", "jumlah": "Jumlah Nasabah"})
fig.show()

rate = master.groupby("jenis_kredit_diajukan")["label"].apply(lambda s: (s=="Diterima").mean()*100).round(1)
print("Insight — approval rate per jenis kredit (%):")
print(rate.to_string())


Insight — approval rate per jenis kredit (%):
jenis_kredit_diajukan
KI     84.3
KMK    85.0
KUR    85.2


**4.2.8 BARU — Nominal Diajukan vs Label**

In [26]:
fig = px.box(master, x="label", y="loan_requested", color="label",
             color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
             title="Nominal Kredit Diajukan vs Label",
             labels={"loan_requested": "Nominal Diajukan (IDR)", "label": "Label"})
fig.show()

med_loan = master.groupby("label")["loan_requested"].median()
print(f"Insight: median nominal diajukan Diterima Rp{med_loan.get('Diterima', 0):,.0f} vs "
      f"Ditolak Rp{med_loan.get('Ditolak', 0):,.0f}.")


Insight: median nominal diajukan Diterima Rp200,000,000 vs Ditolak Rp200,000,000.


**4.2.9 Total Current Balance vs Label**

In [27]:
fig = px.box(master, x="label", y="bank_current_balance_total", color="label",
             color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
             log_y=True,
             title="Total Current Balance vs Label (log scale)",
             labels={"bank_current_balance_total": "Total Current Balance (IDR)", "label": "Label"})
fig.show()

med_bal = master.groupby("label")["bank_current_balance_total"].median()
print(f"Insight: median saldo Diterima Rp{med_bal.get('Diterima', 0):,.0f} vs "
      f"Ditolak Rp{med_bal.get('Ditolak', 0):,.0f} — tidak ada perbedaan besar antar kelompok label.")


Insight: median saldo Diterima Rp18,442,442 vs Ditolak Rp18,790,938 — tidak ada perbedaan besar antar kelompok label.


**4.2.10 Approval Rate per Cabang (lewat RM)**

In [28]:
rm_summary = master.groupby(["rm_branch_name"]).agg(
    jumlah_nasabah=("NIK", "count"),
    approval_rate=("label", lambda s: (s == "Diterima").mean() * 100)
).reset_index().sort_values("approval_rate")

rm_summary["approval_rate"] = rm_summary["approval_rate"].round(1)

fig = px.bar(rm_summary, x="rm_branch_name", y="approval_rate", text="approval_rate",
             title="Approval Rate per Cabang (berdasarkan RM yang menangani)",
             labels={"rm_branch_name": "Cabang RM", "approval_rate": "Approval Rate (%)"})
fig.add_hline(y=(master["label"]=="Diterima").mean()*100, line_dash="dash", line_color="gray",
              annotation_text="rata-rata keseluruhan")
fig.show()

print("Insight — approval rate per cabang (%), diurutkan dari terendah:")
print(rm_summary.to_string(index=False))


Insight — approval rate per cabang (%), diurutkan dari terendah:
         rm_branch_name  jumlah_nasabah  approval_rate
     KCP Depok Margonda             294           81.6
             KCP Cikini             293           82.3
             KCP Kemang             312           83.7
       KCP Bekasi Barat             309           83.8
KCP Bogor Baranangsiang             272           85.3
              KCP Pluit             315           85.4
      KCP Tangerang BSD             279           86.0
            KCP Cibubur             330           86.1
      KCP Kelapa Gading             314           86.6
              KCP Tebet             282           87.2


### 4.3 Korelasi Antar Fitur Numerik

In [29]:
numeric_cols = ["owner_age","business_age_year","employee_count","monthly_turnover_est",
                "loan_requested","tenor_diajukan_bulan","bunga_persen_diajukan",
                "cicilan_bulanan_pengajuan","loan_to_annual_revenue",
                "collateral_ratio","collateral_size_m2","estimated_dsr",
                "slik_worst_collectability","slik_n_loans","revenue_growth_pct",
                "profit_margin_2025","liability_to_asset_2025","bank_best_avg_balance_6m",
                "bank_current_balance_total","bank_total_overdraft_6m"]
corr = master[numeric_cols].corr().round(2)

fig = px.imshow(corr, text_auto=True, aspect="auto", color_continuous_scale="RdBu_r",
                 zmin=-1, zmax=1, title="Korelasi Antar Fitur Numerik Utama (termasuk fitur pengajuan kredit)")
fig.update_layout(height=750)
fig.show()

print("Insight — korelasi terkuat yang perlu dihighlight:")
corr_pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().sort_values(key=abs, ascending=False)
print(corr_pairs.head(8))


Insight — korelasi terkuat yang perlu dihighlight:
loan_requested             cicilan_bulanan_pengajuan     0.83
bank_best_avg_balance_6m   bank_current_balance_total    0.82
monthly_turnover_est       bank_best_avg_balance_6m      0.72
loan_requested             loan_to_annual_revenue        0.69
monthly_turnover_est       bank_current_balance_total    0.59
cicilan_bulanan_pengajuan  loan_to_annual_revenue        0.57
slik_worst_collectability  slik_n_loans                  0.49
loan_requested             collateral_ratio             -0.47
dtype: float64


## 5. Agentic Screening Pipeline

7 agent (Identity, Credit History, DHN, Collateral, Financial, Cashflow,
Risk sebagai orkestrator) — logic TIDAK diubah dari versi sebelumnya.

In [30]:
def identity_agent(row):
    valid_nik = len(str(row["NIK"])) == 16
    valid_age = row["owner_age"] >= 21
    passed = valid_nik and valid_age
    notes = []
    if not valid_nik: notes.append("Format NIK tidak valid")
    if not valid_age: notes.append("Usia pemohon di bawah 21 tahun")
    return pd.Series({
        "identity_passed": passed,
        "identity_notes": "; ".join(notes) if notes else "Identitas valid",
    })

def credit_history_agent(row):
    if row["slik_has_credit_history"] == 0:
        score = 0.6
        notes = "Belum memiliki riwayat kredit di SLIK (nasabah baru)"
    else:
        collect = row["slik_worst_collectability"]
        score = {1: 1.0, 2: 0.75, 3: 0.45, 4: 0.2, 5: 0.0}.get(int(collect), 0.5)
        label_map = {1:"Lancar",2:"Dalam Perhatian Khusus",3:"Kurang Lancar",4:"Diragukan",5:"Macet"}
        notes = f"Kolektibilitas terburuk: {label_map.get(int(collect))} di {int(row['slik_n_banks'])} bank"
        if row["slik_has_macet"] == 1:
            notes += " — riwayat Macet ditemukan"
    return pd.Series({"character_score": round(score, 3), "character_notes": notes})

def dhn_agent(row):
    blacklisted = row["status_dhn"] == "Ya"
    notes = row["dhn_alasan"] if blacklisted else "Tidak terdaftar di Daftar Hitam Nasional"
    return pd.Series({"dhn_blacklisted": blacklisted, "dhn_notes": notes})

def collateral_agent(row):
    ratio = row["collateral_ratio"]
    match = row["ownership_match"] == "Ya"
    score = np.clip(ratio / 1.5, 0, 1)
    if not match:
        score = min(score, 0.2)
    notes = f"LTV agunan {ratio*100:.0f}% dari pinjaman"
    if not match:
        notes += " — nama sertifikat TIDAK sesuai pemilik (butuh verifikasi manual)"
    return pd.Series({"collateral_score": round(float(score), 3), "collateral_notes": notes})

def financial_agent(row):
    growth = row["revenue_growth_pct"]
    margin = row["profit_margin_2025"]
    score = np.clip(0.5 + growth, 0, 1) * 0.6 + np.clip(margin / 0.15, 0, 1) * 0.4
    trend = "tumbuh" if growth > 0.02 else ("stagnan" if growth > -0.02 else "menurun")
    notes = f"Omset {trend} {growth*100:+.1f}% (2024→2025), margin laba {margin*100:.1f}%"
    return pd.Series({"financial_score": round(float(np.clip(score,0,1)), 3), "financial_notes": notes})

BALANCE_RATIO_P90 = 19.2
CURRENT_RATIO_P90 = 22.8

def cashflow_agent(row):
    monthly_turnover = max(row["monthly_turnover_est"], 1)
    balance_ratio = row["bank_best_avg_balance_6m"] / monthly_turnover
    current_ratio = row["bank_best_current_balance"] / monthly_turnover

    balance_score = np.clip(balance_ratio / BALANCE_RATIO_P90, 0, 1)
    current_score = np.clip(current_ratio / CURRENT_RATIO_P90, 0, 1)
    base_score = 0.5 * balance_score + 0.5 * current_score

    overdraft_penalty = min(row["bank_total_overdraft_6m"] * 0.1, 0.3)
    dormant_penalty = 0.15 if row["bank_any_dormant"] == 1 else 0
    score = np.clip(base_score - overdraft_penalty - dormant_penalty, 0, 1)

    notes = (f"Saldo rata-rata {balance_ratio:.1f}x omset bulanan, "
             f"saldo real-time {current_ratio:.1f}x omset bulanan")
    if row["bank_total_overdraft_6m"] > 0:
        notes += f", overdraft {int(row['bank_total_overdraft_6m'])}x dalam 6 bulan"
    if row["bank_any_dormant"] == 1:
        notes += ", memiliki rekening dormant"
    return pd.Series({"cashflow_score": round(float(score), 3), "cashflow_notes": notes})

INDUSTRY_RISK_PENALTY = {"Perdagangan":0.02,"Kuliner":0.04,"Jasa":0.02,
                          "Manufaktur":0.03,"Pertanian":0.06,"Transportasi":0.05}
INTEREST_BY_ZONE = {"Hijau": 9.5, "Kuning": 12.0, "Merah": 15.0}
STRONG, WEAK = 0.7, 0.5

def _compose_insight(decision, scores, score, hard_rule_reason=None):
    weak = [k for k, v in scores.items() if v < WEAK]
    strong = [k for k, v in scores.items() if v >= STRONG]

    if hard_rule_reason:
        return f"Tidak layak karena {hard_rule_reason}."

    if decision == "Layak":
        if weak:
            return (f"Layak tapi {', '.join(weak)} tergolong lemah (skor di bawah 0.5) "
                     "— disarankan tetap dimonitor meski keputusan akhir disetujui.")
        return (f"Layak karena seluruh komponen (" +
                ", ".join(f"{k} {v:.2f}" for k, v in scores.items()) +
                ") berada di zona aman.")

    if decision == "Layak Bersyarat":
        alasan = weak if weak else ["beberapa indikator berada di batas ambang"]
        return f"Layak bersyarat karena {', '.join(alasan)} — disarankan tambahan agunan/penjamin atau plafon diturunkan."

    if decision == "Perlu Review Ulang":
        return (f"Perlu review ulang karena skor gabungan ({score:.2f}) berada di area abu-abu "
                 "— disarankan OTS/wawancara lanjutan sebelum keputusan final.")

    if strong:
        return (f"Tidak layak tapi {', '.join(strong)} tergolong kuat — skor gabungan "
                 f"({score:.2f}) masih di bawah ambang, bisa dipertimbangkan ulang jika "
                 "ada mitigasi risiko dari sisi lain.")
    return f"Tidak layak karena skor gabungan ({score:.2f}) di bawah ambang batas kelayakan pada hampir seluruh komponen."


def risk_agent(row):
    if row["identity_passed"] == False:
        return pd.Series({
            "decision": "Tidak Layak", "zone": "Merah",
            "jenis_kredit_rekomendasi": "-", "nominal_disetujui": 0,
            "jangka_waktu_bulan": 0, "bunga_persen": None, "risk_score": None,
            "insight": _compose_insight("Tidak Layak", {}, 0, row["identity_notes"].lower()),
        })
    if row["dhn_blacklisted"]:
        return pd.Series({
            "decision": "Tidak Layak", "zone": "Merah",
            "jenis_kredit_rekomendasi": "-", "nominal_disetujui": 0,
            "jangka_waktu_bulan": 0, "bunga_persen": None, "risk_score": None,
            "insight": _compose_insight("Tidak Layak", {}, 0,
                f"nasabah terdaftar di Daftar Hitam Nasional ({row['dhn_notes']})"),
        })
    if row["character_score"] == 0.0:
        return pd.Series({
            "decision": "Tidak Layak", "zone": "Merah",
            "jenis_kredit_rekomendasi": "-", "nominal_disetujui": 0,
            "jangka_waktu_bulan": 0, "bunga_persen": None, "risk_score": None,
            "insight": _compose_insight("Tidak Layak", {}, 0, "memiliki riwayat kredit Macet pada SLIK"),
        })

    industry_penalty = INDUSTRY_RISK_PENALTY.get(row["industry"], 0.03)
    condition_score = np.clip(1 - industry_penalty * 4, 0, 1)
    score = (0.35 * row["character_score"] + 0.25 * row["financial_score"]
             + 0.20 * row["collateral_score"] + 0.10 * row["cashflow_score"]
             + 0.10 * condition_score)

    if score >= 0.70:
        decision, zone = "Layak", "Hijau"
    elif score >= 0.55:
        decision, zone = "Layak Bersyarat", "Kuning"
    elif score >= 0.40:
        decision, zone = "Perlu Review Ulang", "Kuning"
    else:
        decision, zone = "Tidak Layak", "Merah"

    # --- Rekomendasi jenis/tenor/nominal MEMPERTIMBANGKAN pengajuan debitur ---
    # (sebelumnya jenis & tenor rekomendasi murni dari ambang loan_requested;
    # sekarang kalau pengajuan debitur sendiri sudah realistis/konsisten,
    # rekomendasi sistem cenderung mengikuti apa yang diajukan)
    max_by_collateral = row["collateral_market_value"] * 0.7
    nominal = int(min(row["loan_requested"], max_by_collateral)) if decision != "Tidak Layak" else 0
    jenis = row["jenis_kredit_diajukan"] if decision != "Tidak Layak" else "-"
    tenor = int(row["tenor_diajukan_bulan"]) if decision != "Tidak Layak" else 0
    bunga = INTEREST_BY_ZONE[zone] if decision != "Tidak Layak" else None

    scores = {"Character": row["character_score"], "Financial": row["financial_score"],
              "Collateral": row["collateral_score"], "Cashflow": row["cashflow_score"]}
    insight = _compose_insight(decision, scores, score)

    return pd.Series({
        "decision": decision, "zone": zone,
        "jenis_kredit_rekomendasi": jenis, "nominal_disetujui": nominal,
        "jangka_waktu_bulan": tenor, "bunga_persen": bunga,
        "risk_score": round(float(score), 3), "insight": insight,
    })


**Catatan perubahan v2 pada Risk Agent:** `jenis_kredit_rekomendasi` dan
`jangka_waktu_bulan` sekarang mengikuti `jenis_kredit_diajukan` &
`tenor_diajukan_bulan` (apa yang diajukan debitur), bukan murni ambang
`loan_requested` seperti sebelumnya — supaya rekomendasi sistem bisa
dibandingkan langsung dengan struktur yang diajukan (persis pola
"Requested Structure vs AI Recommendation" di simulasi kamu).

In [31]:
print("Menjalankan Identity Agent...")
master = master.join(master.apply(identity_agent, axis=1))
print("Menjalankan Credit History Agent...")
master = master.join(master.apply(credit_history_agent, axis=1))
print("Menjalankan DHN Agent...")
master = master.join(master.apply(dhn_agent, axis=1))
print("Menjalankan Collateral Agent...")
master = master.join(master.apply(collateral_agent, axis=1))
print("Menjalankan Financial Agent...")
master = master.join(master.apply(financial_agent, axis=1))
print("Menjalankan Cashflow Agent...")
master = master.join(master.apply(cashflow_agent, axis=1))
print("Menjalankan Risk Agent (orkestrator)...")
master = master.join(master.apply(risk_agent, axis=1))

print("\nDistribusi keputusan Agentic Pipeline:")
print(master["decision"].value_counts(normalize=True).round(3))

master[["application_id","company_name","jenis_kredit_diajukan","tenor_diajukan_bulan",
        "decision","zone","jenis_kredit_rekomendasi","nominal_disetujui",
        "jangka_waktu_bulan","bunga_persen","insight"]].head(5)


Menjalankan Identity Agent...
Menjalankan Credit History Agent...
Menjalankan DHN Agent...
Menjalankan Collateral Agent...
Menjalankan Financial Agent...
Menjalankan Cashflow Agent...
Menjalankan Risk Agent (orkestrator)...

Distribusi keputusan Agentic Pipeline:
decision
Layak                 0.688
Layak Bersyarat       0.206
Tidak Layak           0.075
Perlu Review Ulang    0.031
Name: proportion, dtype: float64


,application_id,company_name,jenis_kredit_diajukan,tenor_diajukan_bulan,decision,zone,jenis_kredit_rekomendasi,nominal_disetujui,jangka_waktu_bulan,bunga_persen,insight
0,APP202600001,UD Santoso Abadi,KI,54,Layak,Hijau,KI,300000000,54,9.5,Layak tapi Cashflow tergolong lemah (skor di b...
1,APP202600002,UD Wijaya Mandiri,KUR,24,Layak,Hijau,KUR,75000000,24,9.5,Layak tapi Cashflow tergolong lemah (skor di b...
2,APP202600003,UD Kusuma Sejahtera,KUR,24,Perlu Review Ulang,Kuning,KUR,150000000,24,12.0,Perlu review ulang karena skor gabungan (0.50)...
3,APP202600004,UD Susanto Makmur,KUR,24,Layak,Hijau,KUR,200000000,24,9.5,Layak tapi Cashflow tergolong lemah (skor di b...
4,APP202600005,CV Wijaya Sejahtera,KI,54,Layak,Hijau,KI,750000000,54,9.5,"Layak karena seluruh komponen (Character 0.75,..."


### 5.1 Kategorisasi Kelayakan 4-Level dari `eligibility_score` (Ground Truth)

`kategorikan_kelayakan()` diturunkan dari `eligibility_score` (skor
ground truth Layer 1 ML, dari generator) — dipisah dari `decision` yang
dihasilkan pipeline agent di atas (dari `risk_score`, skala berbeda).
Dua-duanya sengaja dibandingkan di bagian validasi berikutnya.

In [32]:
def kategorikan_kelayakan(eligibility_score):
    if eligibility_score >= 0.80:
        return "Layak"
    elif eligibility_score >= 0.55:
        return "Layak Bersyarat"
    elif eligibility_score >= 0.40:
        return "Perlu Review Ulang"
    else:
        return "Tidak Layak"

master["kategori_kelayakan_ground_truth"] = master["eligibility_score"].apply(kategorikan_kelayakan)

vc = master["kategori_kelayakan_ground_truth"].value_counts().reset_index()
vc.columns = ["kategori", "jumlah"]
order4 = ["Layak", "Layak Bersyarat", "Perlu Review Ulang", "Tidak Layak"]
vc["kategori"] = pd.Categorical(vc["kategori"], categories=order4, ordered=True)
vc = vc.sort_values("kategori")

fig = px.bar(vc, x="kategori", y="jumlah", color="kategori", text_auto=True,
             title="Distribusi Kategori Kelayakan 4-Level (dari eligibility_score, ground truth)",
             labels={"kategori": "Kategori Kelayakan", "jumlah": "Jumlah Nasabah"})
fig.show()

print("Insight:")
print(vc.assign(persen=(vc["jumlah"]/vc["jumlah"].sum()*100).round(1)).to_string(index=False))


Insight:
          kategori  jumlah  persen
             Layak     365    12.2
   Layak Bersyarat    2179    72.6
Perlu Review Ulang     334    11.1
       Tidak Layak     122     4.1


### 5.2 Distribusi 6 Kategori Narasi Insight

In [33]:
def kategori_insight(s):
    for prefix in ["Layak bersyarat karena", "Perlu review ulang karena", "Layak tapi",
                   "Layak karena", "Tidak layak tapi", "Tidak layak karena"]:
        if s.lower().startswith(prefix.lower()):
            return prefix
    return "Lainnya"

master["insight_kategori"] = master["insight"].apply(kategori_insight)
vc = master["insight_kategori"].value_counts().reset_index()
vc.columns = ["kategori", "jumlah"]

order = ["Layak karena", "Layak tapi", "Layak bersyarat karena",
         "Perlu review ulang karena", "Tidak layak tapi", "Tidak layak karena"]
vc["kategori"] = pd.Categorical(vc["kategori"], categories=order, ordered=True)
vc = vc.sort_values("kategori")

fig = px.bar(vc, x="kategori", y="jumlah", text_auto=True,
             title="Distribusi 6 Kategori Narasi Insight",
             labels={"kategori": "Kategori Insight", "jumlah": "Jumlah Nasabah"})
fig.show()

vc2 = vc.copy()
vc2["persen"] = (vc2["jumlah"] / vc2["jumlah"].sum() * 100).round(1)
print("Insight:")
print(vc2.to_string(index=False))


Insight:
                 kategori  jumlah  persen
             Layak karena     759    25.3
               Layak tapi    1306    43.5
   Layak bersyarat karena     618    20.6
Perlu review ulang karena      93     3.1
       Tidak layak karena     224     7.5


### 5.3 Validasi Pipeline vs Label Asli

In [34]:
cm = pd.crosstab(master["decision"], master["label"])
cm = cm.reindex(["Layak", "Layak Bersyarat", "Perlu Review Ulang", "Tidak Layak"])

fig = px.imshow(cm, text_auto=True, aspect="auto", color_continuous_scale="Blues",
                 title="Keputusan Agentic Pipeline (risk_score) vs Label Asli (Ground Truth)",
                 labels={"x": "Label Asli (dari generator)", "y": "Keputusan Agent Pipeline",
                         "color": "Jumlah"})
fig.show()

agree_rate = (master["decision"].isin(["Layak","Layak Bersyarat"]) == (master["label"]=="Diterima")).mean()
print(f"Tingkat kesesuaian keputusan agent pipeline vs label asli: {agree_rate:.1%}")
print()
print("Distribusi zona risiko:")
print(master["zone"].value_counts(normalize=True).round(3))


Tingkat kesesuaian keputusan agent pipeline vs label asli: 92.6%

Distribusi zona risiko:
zone
Hijau     0.688
Kuning    0.237
Merah     0.075
Name: proportion, dtype: float64


In [35]:
cm2 = pd.crosstab(master["kategori_kelayakan_ground_truth"], master["decision"])
cm2 = cm2.reindex(["Layak", "Layak Bersyarat", "Perlu Review Ulang", "Tidak Layak"])

fig = px.imshow(cm2, text_auto=True, aspect="auto", color_continuous_scale="Purples",
                 title="Kategori Kelayakan (dari eligibility_score) vs Keputusan Pipeline (dari risk_score)",
                 labels={"x": "Keputusan Agent Pipeline (risk_score)",
                         "y": "Kategori Ground Truth (eligibility_score)", "color": "Jumlah"})
fig.show()

agree_rate_4cat = (master["kategori_kelayakan_ground_truth"] == master["decision"]).mean()
print(f"Insight: kesesuaian PERSIS 4-kategori antara ground truth & pipeline: {agree_rate_4cat:.1%}")
print("(Wajar tidak 100% karena dua skor ini dihitung dari formula & bobot yang berbeda — "
      "eligibility_score dari generator/Layer1 ground truth, risk_score dari 7-agent pipeline/Layer2.)")


Insight: kesesuaian PERSIS 4-kategori antara ground truth & pipeline: 29.4%
(Wajar tidak 100% karena dua skor ini dihitung dari formula & bobot yang berbeda — eligibility_score dari generator/Layer1 ground truth, risk_score dari 7-agent pipeline/Layer2.)


## 6. Export `master_scored.csv`

Menambahkan 3 kolom pengajuan kredit baru + `kategori_kelayakan_ground_truth`
+ fitur engineering (`cicilan_bulanan_pengajuan`, `loan_to_annual_revenue`,
`kategori_tenor_diajukan`, `kategori_nominal_diajukan`) ke daftar kolom
export, dibanding versi sebelumnya.

In [36]:
export_cols = ["application_id","NIK","company_name","owner_name","industry",
    "sub_industry","province","city","branch_name","region",
    "rm_id","rm_name","rm_branch_name","level",   # kolom RM: monitoring saja, JANGAN dipakai sbg fitur model
    "loan_requested","jenis_kredit_diajukan","tenor_diajukan_bulan","tujuan_penggunaan_kredit",
    "bunga_persen_diajukan","cicilan_bulanan_pengajuan","loan_to_annual_revenue",
    "kategori_tenor_diajukan","kategori_nominal_diajukan",
    "collateral_type","collateral_market_value","collateral_ratio",
    "estimated_dsr","revenue_growth_pct","profit_margin_2025",
    "slik_worst_collectability","status_dhn",
    "bank_best_avg_balance_6m","bank_best_current_balance",
    "identity_passed","character_score","character_notes",
    "collateral_score","collateral_notes","financial_score","financial_notes",
    "cashflow_score","cashflow_notes","risk_score",
    "decision","zone","kategori_kelayakan_ground_truth",
    "jenis_kredit_rekomendasi","nominal_disetujui",
    "jangka_waktu_bulan","bunga_persen","insight","insight_kategori","label"]

master_export = master[export_cols].copy()
master_export.to_csv(f"{PROCESSED_DIR}/master_scored.csv", index=False)

print(f"Tersimpan: {PROCESSED_DIR}/master_scored.csv "
      f"({master_export.shape[0]} baris x {master_export.shape[1]} kolom)")
master_export.head(3)


Tersimpan: ../data/processed/master_scored.csv (3000 baris x 53 kolom)


,application_id,NIK,company_name,owner_name,industry,sub_industry,province,city,branch_name,region,rm_id,rm_name,rm_branch_name,level,loan_requested,jenis_kredit_diajukan,tenor_diajukan_bulan,tujuan_penggunaan_kredit,bunga_persen_diajukan,cicilan_bulanan_pengajuan,loan_to_annual_revenue,kategori_tenor_diajukan,kategori_nominal_diajukan,collateral_type,collateral_market_value,collateral_ratio,estimated_dsr,revenue_growth_pct,profit_margin_2025,slik_worst_collectability,status_dhn,bank_best_avg_balance_6m,bank_best_current_balance,identity_passed,character_score,character_notes,collateral_score,collateral_notes,financial_score,financial_notes,cashflow_score,cashflow_notes,risk_score,decision,zone,kategori_kelayakan_ground_truth,jenis_kredit_rekomendasi,nominal_disetujui,jangka_waktu_bulan,bunga_persen,insight,insight_kategori,label
0,APP202600001,3276010601750001,UD Santoso Abadi,Budi Panjaitan,Manufaktur,Konveksi,Jawa Barat,Depok,KCP Bogor Baranangsiang,Region 2,RM0020,Indah Kusuma,KCP Bogor Baranangsiang,Junior RB,300000000,KI,54,Pembelian mesin produksi tambahan usaha Konveksi,10.0,8055556.0,9.356,Panjang (>36 bln),50-500jt (KUR Kecil/KMK/KI),Rumah,2328496000,7.76,3.0,0.0809,0.1035,1.0,Tidak,19800614,13581790,True,1.00,Kolektibilitas terburuk: Lancar di 1 bank,1.0,LTV agunan 776% dari pinjaman,0.625,"Omset tumbuh +8.1% (2024→2025), margin laba 10.3%",0.304,"Saldo rata-rata 7.4x omset bulanan, saldo real...",0.825,Layak,Hijau,Layak,KI,300000000,54,9.5,Layak tapi Cashflow tergolong lemah (skor di b...,Layak tapi,Diterima
1,APP202600002,3172010301920002,UD Wijaya Mandiri,Andi Hidayat,Jasa,Bengkel,DKI Jakarta,Jakarta Utara,KCP Bekasi Barat,Region 1,RM0010,Indah Rahman,KCP Bekasi Barat,Senior RB,75000000,KUR,24,Tambahan modal kerja usaha Bengkel,6.0,3500000.0,3.457,Menengah (13-36 bln),50-500jt (KUR Kecil/KMK/KI),Rumah,3724038000,49.65,3.0,-0.0292,0.1199,2.0,Tidak,4014256,1718267,True,0.75,Kolektibilitas terburuk: Dalam Perhatian Khusu...,1.0,LTV agunan 4965% dari pinjaman,0.602,"Omset menurun -2.9% (2024→2025), margin laba 1...",0.079,"Saldo rata-rata 2.2x omset bulanan, saldo real...",0.713,Layak,Hijau,Layak Bersyarat,KUR,75000000,24,9.5,Layak tapi Cashflow tergolong lemah (skor di b...,Layak tapi,Diterima
2,APP202600003,3671010604800003,UD Kusuma Sejahtera,Doni Pratama,Transportasi,Ekspedisi Kecil,Banten,Tangerang,KCP Cibubur,Region 3,RM0039,Slamet Hutapea,KCP Cibubur,Senior RB,150000000,KUR,24,Tambahan modal kerja usaha Ekspedisi Kecil,6.0,7000000.0,7.358,Menengah (13-36 bln),50-500jt (KUR Kecil/KMK/KI),Tanah,1681700000,11.21,3.0,-0.1476,0.1801,4.0,Tidak,14489012,3156322,True,0.20,Kolektibilitas terburuk: Diragukan di 2 bank,1.0,LTV agunan 1121% dari pinjaman,0.611,"Omset menurun -14.8% (2024→2025), margin laba ...",0.013,"Saldo rata-rata 8.5x omset bulanan, saldo real...",0.504,Perlu Review Ulang,Kuning,Perlu Review Ulang,KUR,150000000,24,12.0,Perlu review ulang karena skor gabungan (0.50)...,Perlu review ulang karena,Ditolak


**Catatan kolom RM di file export:** `rm_id`, `rm_name`, `rm_branch_name`,
`level` disertakan untuk keperluan **monitoring/dashboard** (mis. halaman
"Kinerja RM"), TAPI tidak pernah dipakai sebagai input ke 7 agent atau
skema skor manapun — konsisten dengan keputusan governance sebelumnya
(mencegah bias/favoritism RM individual terkunci ke sistem screening).

**Catatan `jenis_kredit_diajukan` & `tujuan_penggunaan_kredit`:** kedua
kolom ini bersifat **informational** — tidak masuk ke `compute_label_score()`
(ground truth generator) maupun bobot `risk_agent()` di atas, sesuai
keputusan sebelumnya. Kegunaannya murni untuk analisis konsistensi
pengajuan (mis. cek "Requested Structure" di simulasi AI Recommendation)
dan sebagai fitur kontekstual untuk ML model Layer 1.